# CSAI415 — D1 Report
## PDF-Papers AI Agent: Online Learning & AutoML

| Attribute | Details |
|---|---|
| **Course** | CSAI415 — Special Topics in AI |
| **Deliverable** | D1 — Week 5 (15%) |
| **Repo** | special_topics_AI |

---

This notebook covers both D1 components:
- **Section 1** — River Online Learner + ADWIN Drift Detection
- **Section 2** — AutoML Baseline Retriever *(Abdullah — added after merge)*

---
# Section 1 — River Online Learner

## Motivation

When a user submits a query to the agent (e.g. *"how does attention work?"*), we classify it into one of **9 research topics** before searching the PDF corpus. Knowing the topic ahead of retrieval helps the system route the query to the right subgraph in Neo4j (D3) and improves retrieval precision.

We use **River** — a Python library for incremental machine learning — to build a classifier that:
- Updates after **every single query** with no retraining
- Detects when the query distribution shifts using **ADWIN**
- Adapts the **BM25 vs dense fusion weight** from user feedback

## Components Built

| Class | Purpose | D1 Requirement |
|---|---|---|
| `QueryTopicLearner` | Classifies queries into 9 topics | River component (i) |
| `HybridWeightAdapter` | Adapts BM25/dense fusion weight | River component (ii) |

## QueryTopicLearner Pipeline

```
raw query
  └─► BagOfWords        word count dict {word: count}
  └─► MultinomialNB     incremental Naive Bayes (9 topics)
```

**Why MultinomialNB over SoftmaxRegression?**  
MultinomialNB is mathematically designed for sparse word count features. It achieved **0.77 accuracy** vs **0.38 for SoftmaxRegression** on the same data — more than 2x better on short query text.

**Prequential evaluation:** predict first → then learn. Accuracy is always measured on data the model has not seen yet.

**Two accuracy metrics:**
- **Cumulative accuracy** — overall accuracy since the start
- **Rolling accuracy (last 50)** — recent performance, more honest for an online learner

**ADWIN drift detection:** watches the error stream (0=correct, 1=wrong). When the recent error rate shifts significantly, rebuilds the full pipeline from scratch.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import random
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display

sys.path.insert(0, str(Path().resolve().parent))

from adaptation import (
    QueryTopicLearner,
    QueryFeedback,
    FeedbackEvent,
    HybridWeightAdapter,
    TOPICS,
    plot_prequential,
)

print("Topics:", TOPICS)
print("Total topics:", len(TOPICS))

In [ ]:
# ── Query templates (15 per topic) ───────────────────────────────────────────
# Diverse templates simulate realistic user queries over the PDF corpus.
# In D2 these are replaced by real queries from users.

QUERY_TEMPLATES = {
    "neural_networks": [
        "how do neural networks learn weights",
        "what is backpropagation in deep learning",
        "explain activation functions in neural nets",
        "what is a multilayer perceptron",
        "how does dropout regularization work",
        "what is vanishing gradient problem",
        "explain weight initialization in deep networks",
        "how does batch normalization help training",
        "what are residual connections in neural networks",
        "explain the role of bias in neural networks",
        "how does a feedforward network make predictions",
        "what is the universal approximation theorem",
        "explain overfitting in deep learning models",
        "how does learning rate affect neural network training",
        "what is gradient descent in neural networks",
    ],
    "transformers": [
        "how does the transformer architecture work",
        "what is self attention mechanism",
        "explain positional encoding in transformers",
        "what is BERT and how is it trained",
        "difference between encoder and decoder in transformers",
        "how does multi head attention work",
        "what is the feed forward layer in a transformer",
        "explain GPT architecture and pretraining",
        "how does cross attention work in transformers",
        "what is masked language modeling in BERT",
        "explain the scaled dot product attention formula",
        "how does the transformer handle variable length input",
        "what is a vision transformer ViT",
        "explain layer normalization in transformers",
        "how does T5 differ from BERT",
    ],
    "reinforcement_learning": [
        "what is the bellman equation in RL",
        "explain Q-learning algorithm",
        "how does policy gradient work",
        "what is reward shaping in reinforcement learning",
        "explain exploration vs exploitation tradeoff",
        "what is the Markov decision process",
        "how does deep Q network DQN work",
        "explain actor critic methods in RL",
        "what is proximal policy optimization PPO",
        "how does model based reinforcement learning differ from model free",
        "what is temporal difference learning",
        "explain Monte Carlo methods in reinforcement learning",
        "how does SARSA differ from Q-learning",
        "what is the role of the value function in RL",
        "explain multi agent reinforcement learning",
    ],
    "computer_vision": [
        "how do convolutional neural networks work",
        "what is image segmentation",
        "explain object detection with YOLO",
        "what is transfer learning in vision",
        "how does ResNet solve vanishing gradients",
        "what is feature extraction in CNNs",
        "explain semantic vs instance segmentation",
        "how does pooling work in convolutional networks",
        "what is the role of filters in CNNs",
        "explain data augmentation for image classification",
        "how does optical flow work in video understanding",
        "what is VGGNet and how deep is it",
        "explain the inception module in GoogLeNet",
        "how does image classification differ from object detection",
        "what is keypoint detection in computer vision",
    ],
    "natural_language_processing": [
        "what is word2vec embedding",
        "explain TF-IDF for text retrieval",
        "how does named entity recognition work",
        "what is sentiment analysis",
        "explain tokenization in NLP",
        "what is the difference between stemming and lemmatization",
        "how does machine translation work",
        "explain coreference resolution in NLP",
        "what is text summarization and how is it done",
        "how does GloVe embedding differ from word2vec",
        "explain part of speech tagging",
        "what is dependency parsing in NLP",
        "how does question answering work in NLP",
        "explain the bag of words model for text classification",
        "what is zero shot learning in NLP",
    ],
    "graph_neural_networks": [
        "how do graph neural networks propagate messages",
        "what is node classification in GNNs",
        "explain graph convolutional networks",
        "what is link prediction in knowledge graphs",
        "how does GraphSAGE handle large graphs",
        "what is the difference between GCN and GAT",
        "explain graph attention networks",
        "how does message passing work in graph networks",
        "what is graph pooling and why is it needed",
        "explain the Weisfeiler Lehman graph isomorphism test",
        "how do GNNs handle heterogeneous graphs",
        "what is spectral graph convolution",
        "explain graph classification with neural networks",
        "how does node2vec learn graph embeddings",
        "what is a knowledge graph embedding",
    ],
    "generative_models": [
        "how do GANs generate images",
        "what is a variational autoencoder",
        "explain diffusion models for image generation",
        "how does stable diffusion work",
        "what is mode collapse in GANs",
        "explain the reparameterization trick in VAEs",
        "how does DALL-E generate images from text",
        "what is the discriminator in a GAN",
        "explain conditional image generation",
        "how does normalizing flow work as a generative model",
        "what is the ELBO loss in variational autoencoders",
        "explain score matching in diffusion models",
        "how does PixelCNN generate images autoregressively",
        "what is progressive growing of GANs",
        "explain the latent space in generative models",
    ],
    "optimization": [
        "what is stochastic gradient descent",
        "explain Adam optimizer",
        "what is learning rate scheduling",
        "how does momentum work in optimization",
        "explain batch normalization",
        "what is the difference between SGD and Adam",
        "explain cosine annealing for learning rate",
        "how does RMSProp optimizer work",
        "what is gradient clipping and why is it used",
        "explain second order optimization methods",
        "how does weight decay prevent overfitting",
        "what is the role of momentum in SGD",
        "explain learning rate warmup strategies",
        "how does Adagrad adapt learning rates",
        "what is the loss landscape in deep learning",
    ],
    "other": [
        "what papers should I read about AI safety",
        "find me recent survey papers",
        "what is the best dataset for benchmarking",
        "how to cite papers in this corpus",
        "what are the most cited papers here",
        "find papers published at NeurIPS 2023",
        "what are the key challenges in AI research",
        "show me papers by a specific author",
        "what is the h-index of a researcher",
        "find papers related to my research topic",
        "what conferences publish AI papers",
        "how do I find open access papers",
        "what is peer review in academic publishing",
        "find papers that cite a specific work",
        "what are preprint servers for AI research",
    ],
}


def make_query(topic: str) -> str:
    """Sample a query for a topic with natural variation."""
    base = random.choice(QUERY_TEMPLATES[topic])
    prefixes = [
        "can you explain",
        "I want to understand",
        "please describe",
        "help me understand",
        "what does the literature say about",
        "give me an overview of",
    ]
    if random.random() < 0.35:
        base = f"{random.choice(prefixes)} {base}"
    return base


print(f"Templates: {sum(len(v) for v in QUERY_TEMPLATES.values())} total")
print(f"Example: {make_query('transformers')}")

---
## Phase 1 — Normal Learning

**Setup:** All 9 topics equally likely, correct labels, 800 samples.  
**Expected:** Accuracy rises from ~0.11 (random baseline) to 0.6+ as the model learns topic vocabulary.

In [ ]:
random.seed(42)
learner = QueryTopicLearner(seed=42)
drift_steps = []
prev_resets = learner.n_resets

N_PHASE1 = 800

for _ in range(N_PHASE1):
    topic = random.choice(TOPICS[:-1])
    query = make_query(topic)
    result = learner.learn_one(
        QueryFeedback(query=query, topic=topic, helpful=True)
    )
    if learner.n_resets > prev_resets:
        drift_steps.append(result["step"])
        prev_resets = learner.n_resets

print(f"Phase 1 — {N_PHASE1} samples")
print(f"  Cumulative accuracy : {learner.prequential_acc.get():.3f}")
print(f"  Rolling accuracy    : {learner.rolling_acc.get():.3f}")
print(f"  Random baseline     : {1/9:.3f}")
print(f"  Improvement         : {learner.prequential_acc.get() / (1/9):.1f}x")
print(f"  Resets              : {learner.n_resets}")

---
## Phase 2 — Drift Injection

**Setup:** Only 1 topic dominates (transformers) + 70% label noise.  
**Expected:** Accuracy drops sharply, ADWIN detects the shift and resets the pipeline.

In [ ]:
N_PHASE2 = 400
prev_resets = learner.n_resets
resets_before = learner.n_resets

for _ in range(N_PHASE2):
    query = make_query("transformers")
    topic = random.choice(TOPICS) if random.random() < 0.70 else "transformers"
    result = learner.learn_one(
        QueryFeedback(query=query, topic=topic, helpful=False)
    )
    if learner.n_resets > prev_resets:
        drift_steps.append(result["step"])
        prev_resets = learner.n_resets

print(f"Phase 2 — {N_PHASE2} samples")
print(f"  Cumulative accuracy : {learner.prequential_acc.get():.3f}")
print(f"  Rolling accuracy    : {learner.rolling_acc.get():.3f}")
print(f"  New resets          : {learner.n_resets - resets_before}")
print(f"  Drift steps so far  : {drift_steps}")

---
## Phase 3 — Recovery

**Setup:** New stable distribution with 3 topics, clean labels, 400 samples.  
**Expected:** Rolling accuracy recovers after the drift-triggered reset.

In [ ]:
N_PHASE3 = 400
recovery_topics = ["computer_vision", "optimization", "neural_networks"]
prev_resets = learner.n_resets

for _ in range(N_PHASE3):
    topic = random.choice(recovery_topics)
    query = make_query(topic)
    result = learner.learn_one(
        QueryFeedback(query=query, topic=topic, helpful=True)
    )
    if learner.n_resets > prev_resets:
        drift_steps.append(result["step"])
        prev_resets = learner.n_resets

print(f"Phase 3 — {N_PHASE3} samples")
print(f"  Cumulative accuracy : {learner.prequential_acc.get():.3f}")
print(f"  Rolling accuracy    : {learner.rolling_acc.get():.3f}")
print(f"  Total resets        : {learner.n_resets}")
print(f"  All drift steps     : {drift_steps}")

---
## Prequential Accuracy Chart

Three lines:
- **Gray** — raw cumulative accuracy (every 10 samples)
- **Blue** — smoothed rolling mean (window=20)
- **Green dashed** — rolling accuracy last 50 queries (recent performance)
- **Red dashed lines** — steps where ADWIN detected drift and reset the pipeline

In [ ]:
Path("../docs").mkdir(exist_ok=True)
learner.save("../docs/learner_history.json")

history_dicts = [
    {
        "step":             s.step,
        "accuracy":         s.accuracy,
        "rolling_accuracy": s.rolling_accuracy,
        "drift_detected":   s.drift_detected,
        "resets":           s.resets,
    }
    for s in learner.history
]

plot_prequential(
    history=history_dicts,
    drift_steps=drift_steps if drift_steps else None,
    output_path="../docs/prequential_accuracy.png",
    window=20,
    show_rolling=True,
)

display(Image(filename="../docs/prequential_accuracy.png"))

---
## ADWIN Controlled Demonstration

To clearly show ADWIN works independently of model accuracy, we run a controlled experiment:
- **600 correct predictions** (error=0) → stable low error rate
- **600 wrong predictions** (error=1) → sharp spike

ADWIN detects the **change** in error rate, not just high error.

In [ ]:
from river import drift as river_drift
import numpy as np

adwin = river_drift.ADWIN(delta=0.002)
adwin_fired_at = []
error_stream = [0] * 600 + [1] * 600
error_rates = []
errors_seen = []
window_size = 30

for i, error in enumerate(error_stream):
    adwin.update(error)
    errors_seen.append(error)
    start = max(0, i - window_size + 1)
    error_rates.append(np.mean(errors_seen[start:i+1]))
    if adwin.drift_detected:
        adwin_fired_at.append(i)
        adwin = river_drift.ADWIN(delta=0.002)

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#f8f8f8")
ax.set_facecolor("#f8f8f8")
ax.plot(error_rates, color="#4361ee", linewidth=2,
        label=f"Rolling error rate (window={window_size})")
ax.axvline(x=600, color="orange", linewidth=1.5,
           linestyle="--", alpha=0.8, label="Drift injected at step 600")
for step in adwin_fired_at:
    ax.axvline(x=step, color="#e63946", linewidth=1.5, linestyle="--", alpha=0.8)
red_patch = mpatches.Patch(color="#e63946", alpha=0.8,
                            label=f"ADWIN fired at: {adwin_fired_at}")
handles, labels = ax.get_legend_handles_labels()
handles.append(red_patch)
ax.legend(handles=handles, fontsize=9)
ax.set_xlabel("Steps", fontsize=11)
ax.set_ylabel("Error rate", fontsize=11)
ax.set_title("ADWIN drift detection — controlled error stream", fontsize=12)
ax.set_ylim(-0.05, 1.1)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("../docs/adwin_demo.png", dpi=150)
plt.show()
print(f"ADWIN fired at steps: {adwin_fired_at}")

---
## HybridWeightAdapter — BM25 vs Dense Fusion

The second River component adapts the fusion weight between BM25 (keyword) and dense (vector) retrieval based on user feedback.

- `alpha = 0.0` → pure BM25
- `alpha = 1.0` → pure dense
- `alpha = 0.5` → equal blend (default)

This connects directly to the retrieval pipeline in D2 — the `/feedback` endpoint updates alpha in real time.

In [ ]:
adapter = HybridWeightAdapter(alpha=0.5, lr=0.01)

# Simulate a sequence of user feedback events
feedback_sequence = [
    (True,  "dense"),    # dense worked → alpha increases
    (True,  "dense"),    # dense worked again
    (False, "dense"),    # dense failed → alpha decreases
    (True,  "bm25"),     # bm25 worked → alpha decreases
    (True,  "bm25"),     # bm25 worked again
    (False, "bm25"),     # bm25 failed → alpha increases
    (True,  "hybrid"),   # hybrid worked → no change
    (True,  "dense"),    # dense worked
    (True,  "dense"),    # dense worked
    (True,  "dense"),    # dense worked — alpha drifts up
]

print(f"Initial alpha: {adapter.alpha}")
print(f"{'Step':<6} {'helpful':<10} {'type':<10} {'alpha'}")
print("-" * 36)

for helpful, rtype in feedback_sequence:
    new_alpha = adapter.update(helpful=helpful, retrieval_type=rtype)
    print(f"{adapter.step:<6} {str(helpful):<10} {rtype:<10} {new_alpha:.3f}")

print(f"\nFinal weights: {adapter.get_weights()}")
print("  → dense_weight is passed to the retrieval pipeline in D2")

In [ ]:
# Plot alpha over time
steps  = [h["step"]  for h in adapter.history]
alphas = [h["alpha"] for h in adapter.history]

fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_facecolor("#f8f8f8")
ax.set_facecolor("#f8f8f8")
ax.plot(steps, alphas, color="#4361ee", linewidth=2, marker="o", markersize=5)
ax.axhline(y=0.5, color="gray", linewidth=1, linestyle="--", alpha=0.5, label="Equal blend (0.5)")
ax.set_xlabel("Feedback events", fontsize=10)
ax.set_ylabel("Alpha (dense weight)", fontsize=10)
ax.set_title("HybridWeightAdapter — alpha evolution from user feedback", fontsize=11)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("../docs/hybrid_adapter.png", dpi=150)
plt.show()

---
## Results Summary

In [ ]:
random_baseline = round(1 / len(TOPICS), 3)
final_acc       = round(learner.prequential_acc.get(), 3)
final_rolling   = round(learner.rolling_acc.get(), 3)

summary = {
    "Metric": [
        "Total samples processed",
        "Random baseline (1/9)",
        "Final cumulative accuracy",
        "Final rolling accuracy (last 50)",
        "Improvement over random",
        "Total ADWIN resets",
        "Drift detected at steps",
        "ADWIN controlled demo fired at",
        "HybridWeightAdapter final alpha",
        "Vectorizer",
        "Classifier",
        "Drift detector",
    ],
    "Value": [
        learner.n_samples,
        random_baseline,
        final_acc,
        final_rolling,
        f"{final_acc / random_baseline:.1f}x",
        learner.n_resets,
        str(drift_steps) if drift_steps else "none during simulation",
        str(adwin_fired_at),
        adapter.get_weights(),
        "BagOfWords (incremental vocabulary)",
        "MultinomialNB (alpha=1.0, Laplace smoothing)",
        "ADWIN (delta=0.002)",
    ],
}

df = pd.DataFrame(summary)
df.style.set_properties(**{"text-align": "left"}).hide(axis="index")

---
## Design Decisions & Pitfalls

**MultinomialNB over SoftmaxRegression**  
MultinomialNB is designed for sparse word count features. In testing it achieved 0.77 accuracy vs 0.38 for SoftmaxRegression on the same synthetic data. The key advantage is that NB learns per-topic word probability distributions which aligns perfectly with how scientific queries are structured.

**Two accuracy metrics**  
Cumulative accuracy smooths over the full history and never drops sharply. Rolling accuracy (last 50 queries) shows what the model is doing right now — much more informative for an online learner where recent performance matters more than historical average.

**Full pipeline reset on drift**  
Unlike a selective reset (keep vocabulary, reset classifier), we rebuild the full pipeline on drift. When topic distributions shift completely, the old vocabulary weights are also stale. A full reset allows faster recovery on the new distribution.

**HybridWeightAdapter learning rate**  
lr=0.01 means each feedback nudges alpha by 1%. This is intentionally small — a single unhelpful response should not drastically change the fusion weight. The adapter needs several consistent signals before meaningfully shifting.

**Synthetic data limitation**  
Accuracy on synthetic templates is limited by vocabulary overlap between topics. Real user queries from the corpus (D2) will have richer, more diverse vocabulary and higher accuracy is expected.

**Future improvement**  
In D2, `bge-small-en` will already be loaded for retrieval. Swapping BagOfWords for frozen embeddings is a one-line change and would give better semantic understanding for paraphrased queries.

---
# Section 2 — AutoML Baseline Retriever
> **Abdullah's section — to be completed after `feat/automl-baseline` is merged into main**